## Current Evalution Script:

In [ ]:
# Replacement Evaluation cell (full) with updated Temporal Gain metric per your description
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from collections import Counter
import math
from scipy.signal import correlate

# --------------------------
# Helpers (scaler inversion, glycemia classes, metrics)
# --------------------------
def inv_cgm_with_cg_scaler(y_scaled):
    """y_scaled: array shape (..., N_OUT) or (..., N_OUT, 1) -> returns same shape in mg/dL"""
    if y_scaled is None or np.asarray(y_scaled).size == 0:
        return np.array([]).reshape(0, N_OUT)
    arr = np.asarray(y_scaled)
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr.reshape(arr.shape[0], arr.shape[1])
    flat = arr.reshape(-1, 1)
    inv = cg_scaler.inverse_transform(flat).reshape(-1, N_OUT)
    return inv

HYPO = 70.0; HYPER = 180.0
def glycemia_class(x):
    x = np.asarray(x).reshape(-1)
    cls = np.full(x.shape, -1, dtype=int)
    cls[x < HYPO] = 0
    cls[(x >= HYPO) & (x <= HYPER)] = 1
    cls[x > HYPER] = 2
    return cls

# ---------- Updated Temporal Gain (TG) implementation ----------
def temporal_gain(y_true, y_pred, prediction_horizon):
    """
    Compute Temporal Gain (TG) following the description:
      - For each forecast window (length = horizon steps), compute cross-correlation between measured
        and predicted horizon-length vectors.
      - Restrict allowed delay tau to [0 .. max_lag_steps] where max_lag_steps = prediction_horizon / dt.
      - Find tau that maximizes abs(cross_corr). Convert tau to minutes: tau_minutes = tau * dt.
      - TG_window = prediction_horizon - tau_minutes.
      - Aggregate windows by median and return TG (minutes), clipped to [0, prediction_horizon].
    Inputs:
      y_true: array shape (n_windows, horizon) OR (n_windows*horizon,) that can be reshaped
      y_pred: same shape as y_true
      prediction_horizon: minutes (e.g., 30)
    Returns:
      median_TG_minutes (int)
    Notes:
      - If a window has very low variance (constant), it is considered uninformative and TG_window = 0.
      - This implementation follows the paper's definition TG = PH - delay * dt.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # infer horizon in steps and dt in minutes
    # prefer global N_OUT if available
    if 'N_OUT' in globals():
        horizon = int(N_OUT)
    else:
        # fallback: assume dt = 5 and compute horizon
        horizon = int(prediction_horizon // 5) or 1
    dt_minutes = float(prediction_horizon) / float(horizon)  # e.g., 30/6 = 5

    # reshape if flattened inputs were provided
    if y_true.ndim == 1 and y_pred.ndim == 1 and (y_true.size % horizon == 0) and (y_pred.size % horizon == 0):
        try:
            y_true = y_true.reshape(-1, horizon)
            y_pred = y_pred.reshape(-1, horizon)
        except Exception:
            return int(prediction_horizon)

    # check shapes
    if y_true.ndim != 2 or y_pred.ndim != 2 or y_true.shape[1] != y_pred.shape[1]:
        # cannot compute TG meaningfully
        return int(prediction_horizon)

    n_windows, L = y_true.shape
    max_lag_steps = int(prediction_horizon // dt_minutes)  # should equal horizon typically
    # ensure integer bounds
    max_lag_steps = min(max_lag_steps, L - 1)

    tg_windows = []
    for i in range(n_windows):
        t = y_true[i].astype(float)
        p = y_pred[i].astype(float)

        # Skip windows with NaNs entirely
        if np.all(np.isnan(t)) or np.all(np.isnan(p)):
            tg_windows.append(0)
            continue

        # If true or predicted horizon is (nearly) constant: uninformative for cross-correlation
        if np.nanstd(t) < 1e-8 or np.nanstd(p) < 1e-8:
            # treat as no timing information -> TG = 0 (as per your preference; persistence thus gets 0)
            tg_windows.append(0)
            continue

        # center signals
        t0 = t - np.nanmean(t)
        p0 = p - np.nanmean(p)

        # full cross-correlation and lags: -(L-1) .. (L-1)
        cc = correlate(t0, p0, mode='full')
        full_lags = np.arange(-L + 1, L)

        # allow only non-negative delays up to max_lag_steps
        valid_mask = (full_lags >= 0) & (full_lags <= max_lag_steps)
        valid_lags = full_lags[valid_mask]
        valid_cc   = cc[valid_mask]

        if valid_cc.size == 0:
            # no valid cross-correlation region -> worst-case
            tg_windows.append(0)
            continue

        # find delay (in steps) that maximizes abs(cross-corr)
        idx = np.argmax(np.abs(valid_cc))
        delay_steps = int(valid_lags[idx])
        delay_minutes = delay_steps * dt_minutes
        tg_val = prediction_horizon - delay_minutes
        # clip to [0, prediction_horizon]
        tg_val = max(0.0, min(float(prediction_horizon), float(tg_val)))
        tg_windows.append(tg_val)

    if len(tg_windows) == 0:
        return int(prediction_horizon)
    # aggregate by median for robustness
    tg_median = float(np.median(tg_windows))
    return int(round(tg_median))

#def temporal_gain(y_true, y_pred, prediction_horizon):
    """
    Temporal Gain implementation adapted to behave like Code 1:
      - Flatten inputs and compute a single full cross-correlation.
      - Use fixed dt = 5 minutes (as in Code 1).
      - Consider lags in steps from -prediction_horizon//5 .. 0 (inclusive).
      - Return TG in minutes computed as: TG = prediction_horizon + lag_steps*5
        (since lag_steps are non-positive here).
    Inputs:
      y_true: array-like (1D or 2D)
      y_pred: array-like (1D or 2D)
      prediction_horizon: minutes (e.g., 60)
    Returns:
      TG_minutes (int)
    Notes:
      - This purposely does NOT demean signals or compute per-window TGs to match Code 1.
      - Minimal sanity checks added to avoid runtime errors on empty inputs or empty valid lag range.
    """
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # basic guards
    if y_true.size == 0 or y_pred.size == 0:
        return int(prediction_horizon)

    # full cross-correlation (no demeaning, same as Code 1)
    cc = correlate(y_true, y_pred, mode='full')

    # lag bounds matching Code 1 (dt fixed to 5 minutes)
    lag_min = - (prediction_horizon // 5)
    lag_max = 0

    # full lag indices for correlate output: -(N_true-1) .. (N_pred-1)
    full_lags = np.arange(-len(y_true) + 1, len(y_pred))

    # mask valid lags and pick best (abs(cc))
    valid_mask = (full_lags >= lag_min) & (full_lags <= lag_max)
    if not np.any(valid_mask):
        # fallback if no valid lags (mirror Code 1's lack of handling by returning PH)
        return int(prediction_horizon)

    valid_lags = full_lags[valid_mask]
    valid_cc   = cc[valid_mask]

    # index of maximum absolute cross-correlation
    max_idx = int(np.argmax(np.abs(valid_cc)))
    best_lag_steps = int(valid_lags[max_idx])

    # convert to minutes using dt=5 and compute TG as in Code 1
    tg_minutes = prediction_horizon + best_lag_steps * 5

    # return integer minutes (round to nearest)
    return int(round(tg_minutes))

# --- other metrics unchanged ---
def g_mean(y_true, y_pred):
    from sklearn.metrics import recall_score
    t = glycemia_class(y_true); p = glycemia_class(y_pred)
    recalls = recall_score(t, p, average=None, zero_division=0)
    recalls = np.where(recalls == 0, 1e-10, recalls)
    return float(np.exp(np.mean(np.log(recalls))))

def glucose_weighted_rmse(y_true, y_pred):
    y_true = np.asarray(y_true).flatten(); y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    w = np.maximum(y_true, 0.0)
    if w.sum() == 0:
        w = np.ones_like(w)
    w = w / w.sum()
    return math.sqrt(np.sum(w * (y_true - y_pred)**2))

def glucose_weighted_mae(y_true, y_pred):
    y_true = np.asarray(y_true).flatten(); y_pred = np.asarray(y_pred).flatten()
    if y_true.size == 0:
        return np.nan
    w = np.maximum(y_true, 0.0)
    if w.sum() == 0:
        w = np.ones_like(w)
    w = w / w.sum()
    return np.sum(w * np.abs(y_true - y_pred))

# Clarke/Parkes fallback (fast)
def clarke_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred): return None
    if (ref >= 70 and abs(pred - ref) <= 0.2 * ref) or (ref < 70 and abs(pred - ref) <= 20): return 'A'
    if (pred >= 330 and ref <= 50) or (pred <= 50 and ref >= 330): return 'E'
    if ref < 70 and pred >= 180: return 'C'
    if ref >= 180 and pred < 70: return 'D'
    return 'B'
def parkes_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred): return None
    err = pred - ref; rel_err = abs(err) / max(ref,1.0)
    if (ref < 70 and abs(err) <= 15) or (ref >= 70 and rel_err <= 0.2): return 'A'
    if rel_err <= 0.35: return 'B'
    if (ref < 70 and pred > 180) or (ref > 180 and pred < 70): return 'D'
    if rel_err > 0.6: return 'E'
    return 'C'

# --------------------------
# Prepare batched predictions for each model across all test subjects
# (unchanged from previously working code, updated to include 'pinn')
# --------------------------
models = {
    'persistence': persistence_model['predict'],
    'bergman': bergman_model['predict'],
    'lstm': lambda s: lstm_predict_on_subject(s),
    'ml_corrector': ml_corrector_model['predict'],
    'pinn': pinn_predict_on_subject
}

def ml_corrector_predict_fast(sid):
    info = per_subject_test.get(sid, None)
    if info is None or info['X'].shape[0] == 0:
        return np.zeros((0, N_OUT, 1))
    Xs = info['X']
    resid_scaled = model_corr.predict(Xs, batch_size=128)
    subj = per_subject[sid]
    n_windows = Xs.shape[0]
    G_inits = subj['CGM_smoothed'].values[N_IN-1 : N_IN-1 + n_windows]
    bolus_mat = np.zeros((n_windows, N_OUT))
    carbs_mat  = np.zeros((n_windows, N_OUT))
    for w in range(n_windows):
        start = N_IN + w
        bolus_mat[w, :] = subj['bolus'].values[start:start+N_OUT]
        carbs_mat[w, :] = subj['carbs'].values[start:start+N_OUT]
    p1, p2, p3, k_i, Gb = bergman_model['params']
    dt_scale = (5.0 / 5.0)
    berg_preds = np.zeros((n_windows, N_OUT))
    G = G_inits.copy().astype(float)
    I = np.zeros(n_windows)
    for t in range(N_OUT):
        I = I * math.exp(-k_i) + bolus_mat[:, t]
        meal_effect = p3 * carbs_mat[:, t]
        dG = -p1 * (G - Gb) - p2 * I + meal_effect
        G = G + dG * dt_scale
        berg_preds[:, t] = G
    berg_preds_scaled = cg_scaler.transform(berg_preds.reshape(-1,1)).reshape(-1, N_OUT)
    corrected_scaled = berg_preds_scaled + resid_scaled
    return corrected_scaled.reshape(n_windows, N_OUT, 1)

models['ml_corrector'] = ml_corrector_predict_fast

# --------------------------
# Build arrays across ALL test windows for each model (batched per subject)
# --------------------------
all_results = {}
for mname, mpredict in models.items():
    Ys = []
    for sid, info in per_subject_test.items():
        X_test = info['X']
        if X_test.shape[0] == 0:
            continue
        yhat_scaled = mpredict(sid)
        yhat_scaled = np.asarray(yhat_scaled)
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        yhat_mg = inv_cgm_with_cg_scaler(yhat_scaled)
        Ys.append(yhat_mg)
    all_results[mname] = np.vstack(Ys) if len(Ys) else np.zeros((0, N_OUT))

# Build ground-truth (all windows)
y_true_all_windows = []
for sid, info in per_subject_test.items():
    y_test = info['y']
    if y_test.shape[0] == 0:
        continue
    y_true_mg = inv_cgm_with_cg_scaler(y_test.reshape(-1, N_OUT))
    y_true_all_windows.append(y_true_mg)
if len(y_true_all_windows) == 0:
    raise RuntimeError("No test windows found.")
y_true_all = np.vstack(y_true_all_windows)

# --------------------------
# Metrics (per-horizon and final-step)
# --------------------------
horiz_steps = np.arange(1, N_OUT+1) * 5
print("\n=== Per-horizon RMSE / MAE for each model ===")
per_horizon = {}
for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        print(f"Model {mname}: no predictions")
        continue
    rmse_h = []
    mae_h = []
    for h in range(N_OUT):
        y_t = y_true_all[:, h]
        y_p = ypred_all[:, h]
        rmse_h.append(np.sqrt(mean_squared_error(y_t, y_p)))
        mae_h.append(mean_absolute_error(y_t, y_p))
    per_horizon[mname] = {'rmse': rmse_h, 'mae': mae_h}
    print(f"\nModel: {mname}")
    print("Horizon (min):", list(horiz_steps))
    print("RMSE (mg/dL): ", [round(v,2) for v in rmse_h])
    print("MAE  (mg/dL): ", [round(v,2) for v in mae_h])

print("\n=== Final-step aggregated metrics (30 min) ===")
for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    ref = y_true_all[:, -1]; pred = ypred_all[:, -1]
    rmse_val = np.sqrt(mean_squared_error(ref, pred))
    mae_val  = mean_absolute_error(ref, pred)
    gw_rmse = glucose_weighted_rmse(ref, pred)
    gw_mae  = glucose_weighted_mae(ref, pred)
    # compute TG using multi-step arrays and the paper definition TG = PH - delay*dt (median across windows)
    tg = temporal_gain(y_true_all, ypred_all, prediction_horizon=5*N_OUT)
    gm = g_mean(ref, pred)
    print(f"\nModel: {mname}")
    print(f"  windows: {len(ref)}")
    print(f"  RMSE: {rmse_val:.3f}  MAE: {mae_val:.3f}")
    print(f"  Glucose-weighted RMSE: {gw_rmse:.3f}  Glucose-weighted MAE: {gw_mae:.3f}")
    print(f"  Temporal gain (min): {tg}   G-Mean: {gm:.4f}")

# --------------------------
# Clarke & Parkes (fallback) + scatter plots per model (final step)
# --------------------------
for mname, ypred_all in all_results.items():
    if ypred_all.size == 0:
        continue
    ref = y_true_all[:, -1]; pred = ypred_all[:, -1]
    cz = [clarke_zone_for_point(r,p) for r,p in zip(ref, pred)]
    pz = [parkes_zone_for_point(r,p) for r,p in zip(ref, pred)]
    c_counts = dict(Counter(cz)); p_counts = dict(Counter(pz)); total = len(ref)
    print(f"\n{mname} Clarke zones: ", {k:(c_counts[k], round(100*c_counts[k]/total,2)) for k in c_counts})
    print(f"{mname} Parkes zones : ", {k:(p_counts[k], round(100*p_counts[k]/total,2)) for k in p_counts})

    fig, ax = plt.subplots(1,2, figsize=(12,5))
    ax0 = ax[0]
    ax0.scatter(ref, pred, s=6, alpha=0.5)
    ax0.plot([0,400],[0,400],'--', linewidth=0.8)
    xs = np.linspace(0,400,100); ax0.plot(xs, 1.2*xs, linewidth=0.6); ax0.plot(xs, 0.8*xs, linewidth=0.6)
    ax0.axvline(70, linestyle=':'); ax0.axhline(70, linestyle=':')
    ax0.set_xlim(0,400); ax0.set_ylim(0,400)
    ax0.set_xlabel('Reference (mg/dL)'); ax0.set_ylabel('Predicted (mg/dL)')
    ax0.set_title(f"{mname} — Clarke (final step)")

    ax1 = ax[1]
    ax1.scatter(ref, pred, s=6, alpha=0.5)
    ax1.plot([0,400],[0,400],'--', linewidth=0.8)
    ax1.set_xlim(0,400); ax1.set_ylim(0,400)
    ax1.set_xlabel('Reference (mg/dL)'); ax1.set_ylabel('Predicted (mg/dL)')
    ax1.set_title(f"{mname} — Parkes (visual)")
    plt.suptitle(f"{mname} Pred vs Ref (final step) — N={total}")
    plt.tight_layout()
    plt.show()

# --------------------------
# Glycemia detection confusion matrices for each model (final step)
# --------------------------
def compute_glycemia_confusion(y_true, y_pred):
    y_t = np.asarray(y_true).reshape(-1); y_p = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_t) & np.isfinite(y_p)
    y_t = y_t[mask]; y_p = y_p[mask]
    counts = np.zeros((3,3), dtype=int)
    if y_t.size == 0:
        return counts, np.full_like(counts, np.nan, dtype=float)
    tcls = glycemia_class(y_t); pcls = glycemia_class(y_p)
    for t,p in zip(tcls, pcls):
        if 0<=t<=2 and 0<=p<=2:
            counts[t,p] += 1
    pct = np.full_like(counts, np.nan, dtype=float)
    for i in range(3):
        s = counts[i].sum()
        if s>0:
            pct[i,:] = counts[i,:] / float(s)
    return counts, pct

def plot_confusion_heatmaps(counts, pct, title=""):
    fig, ax = plt.subplots(1,1, figsize=(5,4))
    im = ax.imshow(pct, vmin=0.0, vmax=1.0, cmap='Blues', interpolation='nearest')
    ax.set_xticks(np.arange(3)); ax.set_yticks(np.arange(3))
    labels = ['Pred Hypo\n(<70)','Pred Normo\n(70-180)','Pred Hyper\n(>180)']
    ax.set_xticklabels(labels); ax.set_yticklabels(['True Hypo','True Normo','True Hyper'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    for i in range(3):
        for j in range(3):
            c = counts[i,j]; p = pct[i,j]
            if np.isnan(p):
                txt = "n/a\n(0)"
            else:
                txt = f"{p*100:4.1f}%\n({c})"
            ax.text(j, i, txt, ha='center', va='center', color='black', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

for mname, ypred_all in all_results.items():
    if ypred_all.size == 0: continue
    counts, pct = compute_glycemia_confusion(y_true_all[:, -1], ypred_all[:, -1])
    print(f"\nGlycemia detection — {mname} (final step):")
    plot_confusion_heatmaps(counts, pct, title=f"{mname} — Glycemia detection (final step)")

# --------------------------
# UPDATED PLOTTING SECTION (very simple continuous multi-step trajectories)
# Now includes PINN in the panels.
# --------------------------
model_order = ['persistence','bergman','lstm','ml_corrector','pinn']
n_models = len(model_order)
selected_sids = list(per_subject_test.keys())[:3]  # same first 3 test subjects
t_h = np.arange(1, N_OUT+1) * 5  # [5,10,...,30] minutes ahead

if len(selected_sids) == 0:
    print("No test subjects available for plotting.")
else:
    for sid in selected_sids:
        info = per_subject_test[sid]
        X_test = info['X']        # shape (n_windows, N_IN, feats)
        y_test = info['y']        # shape (n_windows, N_OUT, 1) or (n_windows, N_OUT)
        if X_test.shape[0] == 0:
            continue

        # select the last available window (most recent) to produce a single continuous forecast
        last_idx = -1

        # prepare actual multi-step targets (mg/dL) and pick last window's trajectory
        y_test_arr = np.asarray(y_test)
        if y_test_arr.ndim == 3 and y_test_arr.shape[-1] == 1:
            y_test_flat_scaled = y_test_arr.reshape(y_test_arr.shape[0], y_test_arr.shape[1])
        else:
            y_test_flat_scaled = y_test_arr
        y_test_mg = inv_cgm_with_cg_scaler(y_test_flat_scaled)  # (n_windows, N_OUT)
        actual_traj = y_test_mg[last_idx]                      # (N_OUT,)

        # create 1xN figure (N models)
        fig, axs = plt.subplots(1, n_models, figsize=(5*n_models, 4.5), sharey=True)
        fig.suptitle(f"Subject {sid} — Continuous multi-step trajectories (last-window forecast)", fontsize=14)

        for i_m, mname in enumerate(model_order):
            ax = axs[i_m]

            # get model predictions for this subject (batched), then take last window's prediction
            yhat_scaled_all = models[mname](sid)
            yhat_scaled_all = np.asarray(yhat_scaled_all)
            if yhat_scaled_all.size == 0:
                ax.text(0.5, 0.5, "no preds", ha='center', va='center')
                ax.set_title(mname)
                continue
            if yhat_scaled_all.ndim == 3 and yhat_scaled_all.shape[-1] == 1:
                yhat_scaled_all = yhat_scaled_all.reshape(yhat_scaled_all.shape[0], yhat_scaled_all.shape[1])
            yhat_mg_all = inv_cgm_with_cg_scaler(yhat_scaled_all)  # (n_windows, N_OUT)
            pred_traj = yhat_mg_all[last_idx]                      # (N_OUT,)

            # plot actual multi-step trajectory (black '-o-')
            ax.plot(t_h, actual_traj, '-o', color='black', markersize=6, linewidth=1.4, label='Actual (last-window)')
            # plot predicted multi-step trajectory (colored dashed)
            if mname == 'persistence':
                color = 'tab:orange'
            elif mname == 'bergman':
                color = 'tab:green'
            elif mname == 'lstm':
                color = 'tab:blue'
            elif mname == 'ml_corrector':
                color = 'tab:red'
            elif mname == 'pinn':
                color = 'purple'
            else:
                color = 'gray'
            ax.plot(t_h, pred_traj, '--', color=color, linewidth=2.0, label='Predicted (last-window)')

            # optionally also plot subject-level median prediction (thin)
            med = np.nanmedian(yhat_mg_all, axis=0)
            ax.plot(t_h, med, ':', color='gray', linewidth=1.0, label='Subject median')

            ax.set_title(mname)
            ax.set_xlabel('Minutes ahead')
            if i_m == 0:
                ax.set_ylabel('Glucose (mg/dL)')
            ax.set_xticks(t_h)
            ax.set_xlim(0.5, t_h[-1] + 1)
            ax.axhline(HYPO, color='gray', linestyle=':', linewidth=0.8)
            ax.set_ylim(40, 300)
            ax.grid(True, linestyle=':')
            ax.legend(fontsize=8)

        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

# Combined summary plot: per-horizon RMSE averaged across subjects (unchanged)
model_names = list(all_results.keys())
per_subject_rmse = {m: [] for m in model_names}
for sid, info in per_subject_test.items():
    y_test = info['y']
    if y_test.shape[0] == 0:
        continue
    y_true_mg = inv_cgm_with_cg_scaler(y_test.reshape(-1, N_OUT))  # (n_windows, N_OUT)
    for m in model_names:
        yhat_scaled = models[m](sid)
        if np.asarray(yhat_scaled).size == 0:
            continue
        yhat_scaled = np.asarray(yhat_scaled)
        if yhat_scaled.ndim == 3 and yhat_scaled.shape[-1] == 1:
            yhat_scaled = yhat_scaled.reshape(yhat_scaled.shape[0], yhat_scaled.shape[1])
        yhat_mg = inv_cgm_with_cg_scaler(yhat_scaled)
        rmse_per_h = [np.sqrt(mean_squared_error(y_true_mg[:,h], yhat_mg[:,h])) for h in range(N_OUT)]
        per_subject_rmse[m].append(rmse_per_h)

plt.figure(figsize=(8,5))
for m in model_names:
    subj_rmses = np.array(per_subject_rmse[m]) if len(per_subject_rmse[m])>0 else np.zeros((0, N_OUT))
    if subj_rmses.size == 0:
        continue
    mean_rmse = np.nanmean(subj_rmses, axis=0)
    plt.plot(t_h, mean_rmse, '-o', label=m, linewidth=2)
plt.xticks(t_h)
plt.xlabel('Minutes ahead'); plt.ylabel('RMSE (mg/dL)')
plt.title('Per-horizon RMSE averaged across test subjects')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

print("\nDone. Notes:")
print("- TG now follows TG = PH - delay*dt (median across windows).")
print("- For windows with no timing information (constant signals) TG is set to 0 (persistence will return TG~0).")
print("- Higher TG (closer to PH) means better timing alignment; lower TG means larger timing misalignment.")
